# NB7 — IMU (Xsens MTw) 검증 **[CPU]**
`scripts/imu_validate.py`

- **IMU 무릎각** = SMPL 과 같은 정의 (thigh 장축 vs shank 장축 사이각, 펴면 0°). 정지 기립 2s 로 센서→분절 장축 캘리브레이션.
- **샘플링 100 Hz** (파일명 '60fps' 와 무관 — 샘플 수 / 기록 시간으로 실측).
- **동기화**: ① coarse = 영상 전체 motion envelope (NB1) ↔ IMU shank 각속도 envelope 상호상관 (속도 계단·정지 구간 덕에 유일해) ② fine = 무릎각 곡선 상관 (coarse ±1.5s). 무릎각만으로는 ±1 주기 모호 → coarse 필수.
- **지표**: RMSE, bias, bias 제거 RMSE, MAE, max, r, ROM, peak — near(R) / far(L) / far MEASURED vs INTERPOLATED. 판정 RMSE < 3°.
- 비교 가능 관절 = **무릎** (골반 센서 없음 → 고관절 불가). 발목은 확장 예정.

IMU txt 위치: `{ROOT}/raw/imu/MT_01200295_{세션}-000_{DeviceId}.txt`

| SESSION | 영상(.r3d) | IMU |
|---|---|---|
| `0729_walk` | 002 보행과 같은 시각의 r3d | `MT_01200295_002_60fps_walking` |
| `0729_full` | 001 과 같은 시각의 r3d | `MT_01200295_001_full` |

같은 영상에서 다른 구간(window)을 또 돌리려면 `SESSION` 이름만 바꿔 (`0729_walk_w2` 등) NB1 부터 다시.

In [ ]:
#@title 0. 설정 — 세션마다 여기만 바꾸면 됨
ROOT    = "/content/drive/MyDrive/walking_0729"   #@param {type:"string"}
SESSION = "0729_full_3kmh"                        #@param {type:"string"}

from google.colab import drive
drive.mount("/content/drive")
import os, sys, importlib
os.environ["WALK_ROOT"] = ROOT
os.environ["WALK_SESSION"] = SESSION
sys.path.insert(0, f"{ROOT}/scripts")
import walkcfg; importlib.reload(walkcfg)
assert os.path.isdir(f"{ROOT}/scripts"), f"{ROOT}/scripts 없음 — colab_kit 업로드 위치 확인"
print("세션 폴더:", walkcfg.SESS)

In [ ]:
#@title 1. IMU 세션
IMU_PREFIX = "MT_01200295_001_full"  #@param ["MT_01200295_001_full", "MT_01200295_002_60fps_walking"]
import glob
print(len(glob.glob(f"{ROOT}/raw/imu/{IMU_PREFIX}-000_*.txt")), "개 txt (6 이어야 함)")

In [ ]:
#@title 2. IMU 단독 점검 (정지 구간 / 무릎각 범위)
!cd "{ROOT}/scripts" && python imu_validate.py --imu-prefix {IMU_PREFIX} --imu-only
from IPython.display import Image, display
display(Image(f"{walkcfg.IMU_OUT}/imu_only_{IMU_PREFIX}.png"))

In [ ]:
#@title 3. 동기화 + 오차
JUMP_FID = 3164   #@param {type:"integer"}
# 동기화 점프 근처 영상 fid (0729_full = 3164, 0 이면 점프 앵커 없이 envelope 상관만)
_j = f"--jump-fid {JUMP_FID}" if JUMP_FID > 0 else ""
!cd "{ROOT}/scripts" && python imu_validate.py --imu-prefix {IMU_PREFIX} {_j}
display(Image(f"{walkcfg.IMU_OUT}/imu_validation.png"))

### 동기화가 의심스러우면
- `[sync 판정] CHECK` 또는 coarse 상관 r 이 낮으면 → `scan_preview.png` 와 IMU 그래프를 보고 같은 사건(보행 시작, 속도 전환)의 시각을 읽어 수동 offset 지정:
  `영상 t + offset = IMU t`

In [ ]:
#@title 4. (선택) 수동 offset
MANUAL_OFFSET_S = 0.0  #@param {type:"number"}
USE_MANUAL = False     #@param {type:"boolean"}
if USE_MANUAL:
    !cd "{ROOT}/scripts" && python imu_validate.py --imu-prefix {IMU_PREFIX} --offset-s {MANUAL_OFFSET_S}
    display(Image(f"{walkcfg.IMU_OUT}/imu_validation.png"))

## 시각화 (NB7)

In [ ]:
#@title 시각화 — NB7 결과 확인 (viz/NB7/  + viz/videos/)
FIDS = "5660 5750"   #@param {type:"string"}
N_EVEN = 0           #@param {type:"integer"}
# 셀 0 을 안 돌렸어도 혼자 동작하도록 (드라이브 마운트 + 세션 설정)
import os, sys, importlib
if "walkcfg" not in globals():
    ROOT = "/content/drive/MyDrive/walking_0729"   # 셀 0 과 같게
    SESSION = "0729_full_3kmh"                     # 셀 0 과 같게
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["WALK_ROOT"] = ROOT; os.environ["WALK_SESSION"] = SESSION
    sys.path.insert(0, f"{ROOT}/scripts"); import walkcfg
print("세션:", walkcfg.SESS)
MAKE_VIDEO = True    #@param {type:"boolean"}
FPS = 15             #@param {type:"number"}
!pip -q install imageio-ffmpeg
!cd "{ROOT}/scripts" && python viz_stages.py --nb 7 --fids {FIDS} --n {N_EVEN}
if MAKE_VIDEO:
    !cd "{ROOT}/scripts" && python viz_video.py --nb 7 --fps {FPS}
from IPython.display import Image, display
import glob, os
for p in sorted(glob.glob(f"{walkcfg.SESS}/viz/NB7/*.png")):
    print(os.path.basename(p)); display(Image(p, width=1100))
from IPython.display import Video
_mp4 = f"{walkcfg.SESS}/viz/videos/NB7_imu.mp4"
if os.path.exists(_mp4):
    print("영상:", _mp4, f"{os.path.getsize(_mp4)/1e6:.1f} MB  (노트북에 넣어 재생 — 확인 후 출력 지우기 권장)")
    display(Video(_mp4, embed=True, width=1100))